# Data source review

Before building the dashboard I need to know whether the two tables in
`Casestudy-Kiwi.com-BusinessAnalyst.xlsx` can actually answer the questions in the brief:
tickets created, tickets resolved, turnaround time, and the differences between languages
and ticket types.

This notebook only checks the data. Cleaning and the dashboard come after.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

FILE = "data/raw/Casestudy-Kiwi.com-BusinessAnalyst.xlsx"

tickets = pd.read_excel(FILE, sheet_name="fact_ticket")
languages = pd.read_excel(FILE, sheet_name="dim_language")

print("tickets:  ", tickets.shape)
print("languages:", languages.shape)
tickets.head(3)

## 1. Missing values

Nothing is missing except the two refund columns, and those gaps are not spread around
randomly — they all sit on the same small group of tickets.

In [ ]:
print("Missing values per column:\n")
print(tickets.isna().sum())
print("\nLanguages table, total missing values:", languages.isna().sum().sum())

In [ ]:
# Which tickets are the missing ones?
no_flags = tickets[tickets["has_accepted_refund"].isna()]
print(no_flags["Ticket Type"].value_counts())
print("\nTheir language_dwid values:", no_flags["language_dwid"].unique())

All 18 are "Helpdesk communication" tickets, and they also carry language ID `-1`, which
the language table maps to "unknown". So it is one odd group of tickets rather than three
separate problems. They are 0.1% of the data, but they need an explicit decision: either
label them as their own category or leave them out, otherwise they quietly disappear from
any chart split by language or type.

## 2. Duplicates

`ticket_id` has to be unique, otherwise every count on the dashboard is too high.

In [ ]:
print("Rows:                       ", len(tickets))
print("Unique ticket_id:           ", tickets["ticket_id"].nunique())
print("Fully duplicated rows:      ", tickets.duplicated().sum())
print("Duplicates ignoring the id: ", tickets.drop(columns="ticket_id").duplicated().sum())

No duplicates of any kind, so tickets can be counted simply by counting rows.

`bid` (the booking) is a different matter — the brief says one booking can raise several
tickets, and that is what the data shows.

In [ ]:
per_booking = tickets.groupby("bid").size()

print("Bookings:                        ", per_booking.count())
print("Bookings with more than 1 ticket:", (per_booking > 1).sum())
print("Most tickets on one booking:     ", per_booking.max())

About 9% of bookings come back more than once. Worth remembering when counting: rows are
tickets, not customers. It is also a useful number in its own right — repeat contact is a
sign of tickets not being fully resolved the first time.

## 3. Joining the language table

The language filter depends on this join working. Two things to confirm: that the language
IDs are unique in the language table, and that every ID used by a ticket actually exists
there.

In [ ]:
print("Language rows:              ", len(languages))
print("Unique language_dwid:       ", languages["language_dwid"].nunique())

used = set(tickets["language_dwid"])
known = set(languages["language_dwid"])
print("Language IDs used:          ", len(used))
print("Used but missing from table:", len(used - known))
print("In the table, never used:   ", len(known - used))

In [ ]:
# Does the join change the number of rows? It should not.
joined = tickets.merge(languages[["language_dwid", "name"]], on="language_dwid", how="left")
print("Rows before the join:", len(tickets))
print("Rows after the join: ", len(joined))

# Bring the language name onto the ticket table.
tickets["language"] = tickets["language_dwid"].map(languages.set_index("language_dwid")["name"])
print("Tickets with no language name:", tickets["language"].isna().sum())

per_language = tickets["language"].value_counts()
print("Languages actually used:      ", len(per_language))
print("Used by fewer than 5 tickets: ", (per_language < 5).sum())

The row count is unchanged, so no ticket was duplicated or lost, and every language ID
resolves to a name. The language table is a full reference list (185 languages, only 40 of
them used), so the filter in the dashboard should be built from the languages that actually
appear in the data.

## 4. Dates

First, how the dates are stored. The two ticket timestamps are proper dates, but the
language table is inconsistent — some of its date columns are plain text.

In [ ]:
print("tickets, creation:  ", type(tickets["creation_timestamp_utc"].iloc[0]).__name__)
print("tickets, resolution:", type(tickets["resolution_timestamp_utc"].iloc[0]).__name__)
print()
for col in ["date_valid_from", "date_valid_to", "inserted_at_utc"]:
    value = languages[col].iloc[0]
    print("languages, %-16s %-9s %s" % (col + ":", type(value).__name__, value))

`date_valid_from` is text while `date_valid_to` is a real date, so comparing them would
need parsing first. It does not affect the dashboard (I only use the language names), but
it is worth noting. Those columns also use placeholder dates — 1899 and 2999 — which just
mean "always valid".

Next, how precise the ticket timestamps are. Opened in Excel or Google Sheets they look
like they only go down to the minute, but that is the cell format hiding the rest. The
stored values go down to the millisecond.

In [ ]:
created = pd.to_datetime(tickets["creation_timestamp_utc"])
resolved = pd.to_datetime(tickets["resolution_timestamp_utc"])

print(created.head(3).tolist())
print("\nTimestamps landing exactly on a whole minute: %.1f%%"
      % (100 * (created.dt.second == 0).mean()))
print("Timezone stored on the timestamps:", created.dt.tz)

More than precise enough for daily or hourly reporting.

No timezone is stored, but the column names say UTC. So every daily figure on the dashboard
is a UTC day, which will not line up with a local working day in every region. Worth stating
on the dashboard rather than leaving the reader to assume.

Now the period the data covers.

In [ ]:
daily = created.dt.date.value_counts().sort_index()
expected_days = pd.date_range(created.min().date(), created.max().date())

print("First ticket:", created.min())
print("Last ticket: ", created.max())
print("Days covered:", len(daily), "of", len(expected_days), "- no gaps" if len(daily) == len(expected_days) else "- gaps!")
print()
print(daily.to_string())

daily.plot(kind="bar", figsize=(10, 3), title="Tickets created per day", color="#4c78a8")
plt.ylabel("tickets")
plt.show()

13 consecutive days with no missing days. The first ticket lands just after midnight on
the first day and the last one just before midnight on the last, so neither end is a half
day that would look like a sudden drop on the chart.

The clearest pattern is the drop between the two weeks — the second week is well below the
first. The quietest single day is a Sunday.

One thing to keep in mind for the dashboard: 13 days is a short window. A 30-day default
view would be mostly empty, and monthly grouping would not mean anything.

## 5. How long tickets take

This is question 3 of the brief, so it gets a closer look.

In [ ]:
tat = (resolved - created).dt.total_seconds() / 60

print(tat.describe().round(1))

tat.plot(kind="hist", bins=31, figsize=(10, 3), title="Turnaround time (minutes)", color="#4c78a8")
plt.xlabel("minutes")
plt.show()

The shape is the problem. A real support queue has lots of quick tickets and a tail of
slow ones. This is flat between 15 and 45 minutes, with nothing outside those limits at all.

Three checks make it clear what is going on.

In [ ]:
print("Shortest:", tat.min(), "minutes")
print("Longest: ", tat.max(), "minutes")
print("Always a whole number of minutes:", (tat % 1 == 0).all())
print("Resolution has the same milliseconds as creation:",
      (created.dt.microsecond == resolved.dt.microsecond).all())

The last line is the giveaway. Every resolution timestamp shares the exact same fraction
of a second as its creation timestamp, which can only happen if it was calculated from it
rather than recorded when the ticket was actually closed. The resolution time is creation
plus a random whole number of minutes between 15 and 45.

**In context, this is expected.** A take-home exercise is not going to ship real customer
support timestamps, so generated values are a sensible thing for the company to provide,
and it is not a reason to change the dashboard design. What it does change is how the
number should be read and presented:

* The turnaround metric still belongs on the dashboard, and the calculation is the same one
  I would run on production data.
* The **value** it shows here is a placeholder, so I would not draw conclusions from it or
  compare it between languages and ticket types as if it were real performance.
* A median is usually the safer choice than an average because it resists extreme values,
  but that advantage does not apply here — there are no extreme values to resist. I would
  still use the median on real data, for the usual reasons.
* Same-day resolution and anything SLA-shaped are calculated from the same number, so they
  carry the same caveat.

I would note this on the dashboard itself, so nobody reads 30 minutes as a real figure.

## 6. Are any tickets still open?

A backlog metric needs unresolved tickets to exist in the data.

In [ ]:
print("Tickets with no resolution date:", resolved.isna().sum())
print("Tickets resolved after the last creation day:",
      (resolved.dt.date > created.dt.date.max()).sum())

Every ticket in the extract is already resolved, so there is no real backlog to show — a
backlog line would sit at zero by construction, not because the team has no open work.

The 10 tickets resolved after the final day matter for a smaller reason: on the last day of
any date range, "resolved" is cut off while "created" is not, so created vs resolved is not
a fair comparison on that specific day.

## 7. Ticket types

`Ticket Type` packs two things into one column: which queue the ticket is in (which is
really the language) and whether it is about a refund. The brief asks for both, so it needs
splitting.

In [ ]:
print(tickets["Ticket Type"].value_counts())

In [ ]:
tickets["queue"] = tickets["Ticket Type"].str.extract(r"Helpdesk (?:- )?(EN|International|JA|KO)")
tickets["request_type"] = tickets["Ticket Type"].str.extract(r"- (Refund|Non-refund) requests")

print(tickets[["Ticket Type", "queue", "request_type"]].drop_duplicates().to_string(index=False))

Two things to handle:

* The 18 "Helpdesk communication" tickets do not fit the pattern and end up with no queue.
* The Japanese and Korean queues are not split into refund / non-refund, so a refund
  breakdown cannot be compared across every queue.

The brief says the two refund flags only apply to refund tickets, which is easy to check
now that the request type is its own column.

In [ ]:
for flag in ["has_accepted_refund", "has_out_of_pocket_refund"]:
    print(flag)
    print(pd.crosstab(tickets["request_type"].fillna("no request type"), tickets[flag]))
    print()

The flags are set on non-refund tickets too - 25 of them for `has_accepted_refund` and 26
for `has_out_of_pocket_refund` - so the flag and the ticket type disagree. Most likely the
flags describe the booking rather than the message, in which case they cannot be counted
per ticket. Not something the dashboard depends on, but worth raising.

## 8. Which field gives the language?

Two columns point at the language, and they do not agree. `language_dwid`, joined from the
language table, tags 3,239 tickets as English that sit in the "International" queue — the
queue that by definition is everything except English. Another 120 English-tagged tickets
sit in the Japanese and Korean queues.

My first thought was that the two fields might be measuring different things: the queue
being the team or the customer's country, and `language_dwid` the language the request was
actually written in. If that were true, both would be correct and I would keep both. Three
checks talked me out of it.

**Check 1 — does the mismatch go both ways?** If these were two real dimensions, I would
expect English speakers handled by the International team *and* Spanish speakers handled by
the EN team.

In [ ]:
by_queue = pd.crosstab(tickets["queue"], tickets["language"] == "English")
by_queue.columns = ["tagged another language", "tagged English"]
by_queue["% English"] = (100 * by_queue["tagged English"] /
                         (by_queue["tagged English"] + by_queue["tagged another language"])).round(1)
by_queue

It only goes one way. The EN queue is 99.9% consistent — just 14 tickets out of 11,666
disagree — while the other three queues are full of English tags. Whatever is going wrong
only ever points at English.

**Check 2 — does the language change within a single booking?** If a customer writes twice
about the same booking, into the same queue, the language should be the same both times.

In [ ]:
per_booking = tickets.groupby("bid").agg(
    n_tickets=("ticket_id", "size"),
    n_queues=("queue", "nunique"),
    n_languages=("language", "nunique"))

one_queue = per_booking[(per_booking["n_tickets"] > 1) & (per_booking["n_queues"] == 1)]
print("Bookings with several tickets in a single queue:", len(one_queue))
print("Of those, the language changes in:              ", (one_queue["n_languages"] > 1).sum())

mixed = tickets[tickets["bid"].isin(one_queue[one_queue["n_languages"] > 1].index)]
print("\nWhat it changes between:")
print(mixed.groupby("bid")["language"].apply(lambda s: " / ".join(sorted(set(s)))).value_counts().head(6))

Always English and something else, never two other languages together. A genuine per-message
language would occasionally give me Spanish and French on one booking. This looks like a
value that falls back to English rather than one that is measured.

**Check 3 — does the mismatch change over time?** A real mix of customer languages should be
fairly steady across two weeks.

In [ ]:
tickets["day"] = created.dt.date
international = tickets[tickets["queue"] == "International"]

by_day = international.groupby("day").agg(
    tickets=("ticket_id", "size"),
    tagged_english=("language", lambda s: (s == "English").sum()))
by_day["% English"] = (100 * by_day["tagged_english"] / by_day["tickets"]).round(1)

# Control: if routing had changed instead, this share would move too.
by_day["% of all tickets in this queue"] = (100 * by_day["tickets"] /
                                            tickets.groupby("day").size()).round(1)
by_day

This is what settled it. The English share of the International queue sits around 38% in
the first week, then jumps to 55% on 4 July and keeps climbing to 65%. Tickets tagged with
a real language nearly halve, while English-tagged tickets go *up* even though total volume
is falling.

The last column rules out the obvious alternative. If the helpdesk had simply started
routing more English customers to the International queue, that share would jump on 4 July
too. It does not — it drifts gently all the way through. The break is in `language_dwid`,
not in the routing.

**Conclusion.** The two fields are not two different dimensions. They are both trying to
describe the language, and `language_dwid` is the unreliable one: it defaults to English,
and it got worse partway through the period. So I use the **queue from `Ticket Type`** as
the language filter — it is stable, it matches the definitions in the brief, and it is
consistent within itself.

This is also worth reporting back to the team on its own. A language field that quietly
started defaulting to English on 4 July would make English look like a growing share of
the workload when nothing of the sort happened.

## 9. Columns that do not add anything

In [ ]:
print("processing_type:", tickets["processing_type"].unique())
print("is_valid:       ", languages["is_valid"].unique())
print("name and description are identical in every row:",
      (languages["name"] == languages["description"]).all())

All three hold a single value or repeat another column, so none of them is worth a filter
or a chart.

## 10. Summary of data quality issues

**Higher priority — affects the core metrics of the dashboard**

1. **Only resolved tickets are included.** Every ticket carries a resolution timestamp, so
   backlog, net ticket change and the resolved-to-created ratio have nothing to measure. A
   backlog line would sit at zero by construction, not because the team has no open work.
   *In a real setting:* review the extraction query, and if needed ask the source team how
   to include tickets that are still open.

2. **The resolved series is cut off at the end of the window.** 10 tickets are resolved
   after the last creation day. On the final day of any date range "created" is complete
   while "resolved" is not, so the two are not comparable on that day.

3. **Short and old history.** 13 days, 26 Jun 2022 (Sunday) to 8 Jul 2022 (Friday). A
   30-day default view would be mostly empty and monthly grouping would mean nothing.
   Expected for an assignment; in real life I would review the extraction and the source.

4. **Turnaround time is synthetic.** Every ticket is resolved between 15 and 45 minutes
   after creation, always a whole number of minutes, with no outliers, no multi-day tickets
   and no zero or negative times. The giveaway is that each resolution timestamp carries
   the identical millisecond as its creation timestamp, which only happens if it was
   calculated rather than recorded. Expected for an assignment. I still build the metric,
   but I label the value on the dashboard so it is not read as performance.

5. **Two conflicting fields describe the language.** `language_dwid` and the queue inside
   `Ticket Type` disagree: 3,239 English-tagged tickets sit in the International queue and
   120 in the Japanese and Korean queues, while the EN queue is 99.9% consistent. The
   mismatch only ever points at English, and it steps from 38% to 55% on 4 July while
   routing stays stable. `language_dwid` defaults to English and degraded partway through
   the period, so I use the queue from `Ticket Type` as the language filter.

6. **Demand falls steadily across the period**, from about 2,100 tickets a day in the first
   week to about 1,050 in the last, with the quietest day a Sunday. The second week
   contains no weekend, so the fall is real rather than a calendar effect. In a real
   setting I would confirm it is genuine before reporting it as a trend.

7. **The 18 "Helpdesk communication" tickets are a group of their own.** They carry
   language "unknown" (-1), null refund flags, and a type that does not split into a queue
   or a request type. Left as they are, they disappear from every chart broken down by
   language or type. Is this type expected, and how should it be reported?

**Lower priority — affects other fields, limits additional insight**

8. **`processing_type` is always "Manual".** Are no tickets handled by automation? More
   likely the extract excludes them, in which case that data would need to come from
   another source.

9. **Refund flags are set on some non-refund tickets** — 25 for `has_accepted_refund` and
   26 for `has_out_of_pocket_refund` — so the flags and the type name disagree. They look
   like they describe the booking rather than the ticket.

10. **The Japanese and Korean queues are not split by refund / non-refund**, so a refund
    breakdown cannot be compared across every queue.

11. **Dates in the language table are stored inconsistently.** `date_valid_from` is text
    while `date_valid_to` is a real date, so comparing them needs parsing first. Both use
    placeholder values (1899 and 2999) meaning "always valid". Does not affect the dashboard.

12. **Two columns in the language table carry nothing.** `is_valid` is true for every row,
    and `name` and `description` are identical in all 185 rows, so the "full language name"
    the brief describes is not actually available.

**Additional observations — not issues, but they inform the design**

13. The language table holds 185 rows (184 languages plus an "unknown" placeholder) but
    only 40 are used, and 16 of those have fewer than 5 tickets. Not a concern, since the
    language filter is built from the queue.

14. **Timestamps are UTC**, so every daily figure is a UTC day. Worth stating on the
    dashboard, as it will not match a local working day in every region.

15. **Timestamps are precise to the millisecond**, although Excel and Google Sheets display
    them truncated to the minute. More than enough for daily or hourly reporting.

16. **About 9% of bookings generate more than one ticket.** Repeat contact is worth
    reporting — it hints at tickets not being resolved first time.

**What I would do next**

Every issue above, turned into something to do. The number in brackets is the issue it
comes from.

*Cleaning and transformation*

1. Split `Ticket Type` into queue and request type, and give the 18 tickets that do not fit
   the pattern their own label so they stay visible in every breakdown. [7]
2. Use the queue as the language dimension and leave `language_dwid` out of the model. [5]
3. Drop the columns that cannot support anything: `processing_type`, `is_valid`,
   `description`, and the language table beyond the name lookup. [8, 12, 13]
4. If the language table is ever needed for more than names, parse `date_valid_from`
   first — it is text, not a date. [11]

*What to build*

5. Tickets created and tickets resolved per day, split by queue and request type. [core]
6. Turnaround time using the median, built exactly as it would be on real data. [4]
7. Repeat contact rate — about 9% of bookings come back — as a bonus insight. [16]
8. Hourly views are available if useful, since the timestamps are precise enough. [15]
9. Leave backlog and net ticket change out, and say why rather than showing a flat zero. [1]

*What to label on the dashboard*

10. Mark turnaround time as a generated figure so it is not read as performance. [4]
11. State that days are UTC and will not match a local working day everywhere. [14]
12. Exclude or flag the final day of the range when comparing created against resolved. [2]
13. Note that the data covers 13 days: default to the full period, and do not offer a
    monthly view. [3]
14. Show the refund split only where it exists — the Japanese and Korean queues do not
    have one. [10]

*Questions for the team that owns the data*

15. Can the extract include tickets that are still open? [1]
16. What changed in `language_dwid` on 4 July, and can the real language be recovered? [5]
17. What is the "Helpdesk communication" type, and why does it carry no language or refund
    flags? [7]
18. Are the refund flags a property of the booking rather than the ticket, and why are they
    set on non-refund tickets? [9]
19. Are automated tickets excluded, given every row says "Manual"? [8]
20. Is the drop in demand across the period genuine, or an artefact of how the data was
    pulled? [6]

## 11. Save the issue list

The dashboard shows this list on its data quality tab, so it is written out here rather than
retyped there. That way the notebook stays the single place these findings are maintained.

In [ ]:
import os

issues = pd.DataFrame([
    (1,  "Higher", "Only resolved tickets are included",
     "Backlog, net change and the resolved-to-created ratio have nothing to measure."),
    (2,  "Higher", "The resolved series is cut off at the end of the window",
     "10 tickets resolve after the last creation day, so the final day of a range is not comparable."),
    (3,  "Higher", "Short and old history",
     "13 days, 26 Jun to 8 Jul 2022. A 30-day default view would be mostly empty."),
    (4,  "Higher", "Turnaround time is synthetic",
     "Resolution is creation plus 15-45 whole minutes, with identical milliseconds. Label it, do not read it as performance."),
    (5,  "Higher", "Two conflicting fields describe the language",
     "language_dwid defaults to English and breaks on 4 July. The queue from Ticket Type is used instead."),
    (6,  "Higher", "Demand falls steadily across the period",
     "About 2,100 tickets a day down to about 1,050. Confirm it is genuine before reporting it as a trend."),
    (7,  "Higher", "18 'Helpdesk communication' tickets are a group of their own",
     "Unknown language, null refund flags, no queue or request type. Labelled Unclassified so they stay visible."),
    (8,  "Lower",  "processing_type is always 'Manual'",
     "Either nothing is automated, or automated tickets are missing from the extract."),
    (9,  "Lower",  "Refund flags are set on some non-refund tickets",
     "25 and 26 tickets respectively. The flags look booking-level, so they cannot be summed per ticket."),
    (10, "Lower",  "Japanese and Korean queues are not split by refund",
     "A refund breakdown is not comparable across every queue."),
    (11, "Lower",  "Dates in the language table are stored inconsistently",
     "date_valid_from is text, date_valid_to is a date. Placeholder values of 1899 and 2999."),
    (12, "Lower",  "Two columns in the language table carry nothing",
     "is_valid is always true, and name and description are identical in all 185 rows."),
    (13, "Observation", "Most of the language table is unused",
     "185 rows, 40 used, 16 of those with fewer than 5 tickets."),
    (14, "Observation", "Timestamps are UTC",
     "Every daily figure is a UTC day and will not match a local working day everywhere."),
    (15, "Observation", "Timestamps are precise to the millisecond",
     "Spreadsheets display them truncated to the minute. Fine for daily or hourly reporting."),
    (16, "Observation", "About 9% of bookings generate more than one ticket",
     "Repeat contact is worth reporting on its own."),
], columns=["n", "priority", "issue", "detail"])

os.makedirs("data/clean", exist_ok=True)
issues.to_csv("data/clean/data_quality_issues.csv", index=False)
print("Saved", len(issues), "issues to data/clean/data_quality_issues.csv")
print(issues["priority"].value_counts().to_string())